# Notebook 10 — Chatbot con ruoli e memoria di conversazione

**Modulo:** 10 — Costruire con le API
**Tempo stimato:** 3-4 ore

## Cosa costruirai

- Una dimostrazione pratica di perché un LLM "dimentica" senza uno storico esplicito
- Un system prompt verticale per un OrderBot di pizzeria, con menu incorporato
- Un ciclo di conversazione che accumula lo storico turno dopo turno
- Un'estrazione finale dell'ordine in JSON, con validazione (TODO da completare)
- Un chatbot da riga di comando completo, con costo accumulato per turno

## Prerequisiti

```bash
pip install openai anthropic python-dotenv
```

Serve un file `.env` nella cartella del progetto con:
```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
```

---
## Cella 0 — Setup: installazione e caricamento delle chiavi

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    if chiave:
        print(f"OK  {nome}: {chiave[:8]}...")
    else:
        print(f"MANCA  {nome}: non trovata. Controlla il tuo file .env")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)
verifica_chiave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)

In [ ]:
from openai import OpenAI
import anthropic
from typing import Literal

cliente_openai = OpenAI(api_key=OPENAI_KEY) if OPENAI_KEY else None
cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_KEY) if ANTHROPIC_KEY else None

# Verifica sempre l'ID modello aggiornato nella documentazione ufficiale del provider.
MODELLI_DEFAULT = {
    "openai": "gpt-5-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}

# Prezzi indicativi in USD per milione di token (esempio illustrativo).
PREZZI = {
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
}

def calcola_costo(tokens_input: int, tokens_output: int, modello: str) -> dict:
    """Calcola il costo in USD di una chiamata, dato il modello usato."""
    if modello not in PREZZI:
        raise ValueError(f"Modello '{modello}' non presente nella tabella prezzi.")
    prezzi = PREZZI[modello]
    costo_input = (tokens_input / 1_000_000) * prezzi["input"]
    costo_output = (tokens_output / 1_000_000) * prezzi["output"]
    return {
        "modello": modello,
        "tokens_input": tokens_input,
        "tokens_output": tokens_output,
        "costo_totale_usd": round(costo_input + costo_output, 6),
    }


def chiama_llm(
    storico: list[dict],
    provider: Literal["openai", "anthropic"] = "openai",
    system_prompt: str = "Sei un assistente utile e conciso. Rispondi sempre in italiano.",
    modello: str | None = None,
    temperature: float = 0.3,
    max_tokens: int = 400,
) -> dict:
    """Wrapper unificato per una chiamata con storico di conversazione.
    `storico` è una lista di messaggi {role, content} SENZA il system prompt
    (che viene gestito separatamente per uniformare le due API, vedi Modulo 08)."""
    modello = modello or MODELLI_DEFAULT[provider]

    if provider == "openai":
        if cliente_openai is None:
            raise RuntimeError("OPENAI_API_KEY non configurata.")
        messaggi = [{"role": "system", "content": system_prompt}] + storico
        risposta = cliente_openai.chat.completions.create(
            model=modello,
            max_tokens=max_tokens,
            temperature=temperature,
            messages=messaggi,
        )
        return {
            "testo": risposta.choices[0].message.content,
            "tokens_input": risposta.usage.prompt_tokens,
            "tokens_output": risposta.usage.completion_tokens,
            "modello": modello,
            "provider": provider,
        }

    elif provider == "anthropic":
        if cliente_anthropic is None:
            raise RuntimeError("ANTHROPIC_API_KEY non configurata.")
        risposta = cliente_anthropic.messages.create(
            model=modello,
            max_tokens=max_tokens,
            temperature=min(temperature, 1.0),
            system=system_prompt,
            messages=storico,
        )
        return {
            "testo": risposta.content[0].text,
            "tokens_input": risposta.usage.input_tokens,
            "tokens_output": risposta.usage.output_tokens,
            "modello": modello,
            "provider": provider,
        }

    else:
        raise ValueError(f"Provider '{provider}' non supportato.")


print("chiama_llm() e calcola_costo() pronte.")

---
## Cella 1 — Dimostrazione: senza storico, il modello dimentica

Facciamo due chiamate separate, ciascuna con un solo messaggio `user`, come se
fossero due richieste HTTP indipendenti (che è esattamente ciò che sono).

**Prima di eseguire:** pensi che il modello ricorderà il nome dato nella prima
chiamata quando lo chiedi nella seconda?

In [ ]:
# Prima chiamata: ci presentiamo
storico_1 = [{"role": "user", "content": "Ciao, mi chiamo Elena."}]
risposta_1 = chiama_llm(storico_1, provider="openai")
print("Turno 1:", risposta_1["testo"])

# Seconda chiamata: NUOVA lista, senza il turno precedente
storico_2 = [{"role": "user", "content": "Come mi chiamo?"}]
risposta_2 = chiama_llm(storico_2, provider="openai")
print("\nTurno 2 (storico separato):", risposta_2["testo"])

**Le tue osservazioni** (modifica questa cella):

- Il modello ha ricordato il nome? ...
- Perché, tecnicamente, non avrebbe potuto ricordarlo? ...

---
## Cella 2 — Ora con lo storico accumulato

Ripetiamo lo stesso scambio, ma questa volta accumuliamo i messaggi nella
stessa lista e la reinviamo per intero ad ogni turno.

In [ ]:
storico = []

storico.append({"role": "user", "content": "Ciao, mi chiamo Elena."})
risposta = chiama_llm(storico, provider="openai")
storico.append({"role": "assistant", "content": risposta["testo"]})
print("Turno 1:", risposta["testo"])

storico.append({"role": "user", "content": "Come mi chiamo?"})
risposta = chiama_llm(storico, provider="openai")
storico.append({"role": "assistant", "content": risposta["testo"]})
print("\nTurno 2 (stesso storico):", risposta["testo"])

print(f"\nLo storico ora contiene {len(storico)} messaggi.")

---
## Cella 3 — Il system prompt dell'OrderBot

Costruiamo un system prompt verticale per un bot che prende ordini in una
pizzeria, seguendo la struttura discussa nella dispensa: ruolo, obiettivo,
processo, vincoli (il menu), tono.

In [ ]:
MENU_PIZZERIA = """
Il menu della pizzeria è il seguente (prezzi in euro, formato grande/media/piccola dove applicabile):
Pizze:
- Margherita: 9.50 / 7.50 / 6.00
- Diavola: 11.00 / 9.00 / 7.00
- Quattro formaggi: 11.50 / 9.50 / 7.50
- Melanzane e provola: 10.50 / 8.50 / 6.50
Contorni:
- Patatine fritte: 4.00 / 3.00
- Insalata mista: 5.50
Extra (aggiungibili a qualunque pizza):
- Mozzarella extra: 1.50
- Funghi: 1.50
- Prosciutto: 2.50
- Olive: 1.00
Bevande:
- Acqua naturale/frizzante: 2.00
- Coca-Cola: 3.00 / 2.00 / 1.50
- Birra artigianale: 4.50
"""

SYSTEM_PROMPT_ORDERBOT = f"""Sei OrderBot, un servizio automatizzato per raccogliere ordini in una pizzeria.
Prima saluti il cliente, poi raccogli l'ordine (pizze, dimensioni, extra),
quindi chiedi se è per asporto o consegna a domicilio. Se è a domicilio, chiedi
l'indirizzo. Alla fine riepiloghi l'ordine e chiedi conferma prima di chiudere.
Rispondi sempre in modo breve, amichevole e colloquiale, in italiano.
Chiarisci sempre dimensione ed eventuali extra prima di considerare un articolo
completo. Non inventare piatti che non sono nel menu.

{MENU_PIZZERIA}
"""

print(SYSTEM_PROMPT_ORDERBOT)

---
## Cella 4 — Conversazione simulata (senza `input()`)

Per rendere il notebook eseguibile end-to-end senza interazione manuale,
simuliamo prima una conversazione con una lista predefinita di messaggi
dell'utente. È lo stesso ciclo che useremo poi nella versione interattiva.

In [ ]:
MESSAGGI_CLIENTE_SIMULATI = [
    "Ciao! Vorrei ordinare una pizza.",
    "Una Diavola grande, con funghi extra.",
    "No grazie, è tutto. È per asporto.",
    "Sì, confermo l'ordine.",
]

storico_ordine: list[dict] = []
costo_accumulato = 0.0

for messaggio_utente in MESSAGGI_CLIENTE_SIMULATI:
    storico_ordine.append({"role": "user", "content": messaggio_utente})

    risultato = chiama_llm(storico_ordine, provider="openai", system_prompt=SYSTEM_PROMPT_ORDERBOT, temperature=0.3)
    storico_ordine.append({"role": "assistant", "content": risultato["testo"]})

    costo_turno = calcola_costo(risultato["tokens_input"], risultato["tokens_output"], risultato["modello"])
    costo_accumulato += costo_turno["costo_totale_usd"]

    print(f"Cliente: {messaggio_utente}")
    print(f"OrderBot: {risultato['testo']}")
    print(f"[costo turno: ${costo_turno['costo_totale_usd']:.6f} | totale: ${costo_accumulato:.6f}]\n")

---
## Cella 5 — Estrazione del riepilogo in JSON

A fine conversazione, chiediamo al modello di generare il riepilogo
dell'ordine in formato strutturato, come discusso nella dispensa (sezione 4).

In [ ]:
ISTRUZIONE_RIEPILOGO = """Genera un riepilogo dell'ordine in formato JSON.
L'output deve essere ESCLUSIVAMENTE il JSON, senza testo aggiuntivo prima o dopo.
Chiavi richieste:
"pizze": lista di oggetti {"nome": str, "dimensione": str, "extra": lista di str, "prezzo": numero}
"contorni": lista di oggetti {"nome": str, "prezzo": numero}
"bevande": lista di oggetti {"nome": str, "dimensione": str o null, "prezzo": numero}
"tipo_consegna": "asporto" oppure "domicilio"
"indirizzo": stringa oppure null se asporto
"totale_stimato": numero
"""

storico_ordine.append({"role": "user", "content": ISTRUZIONE_RIEPILOGO})
risultato_riepilogo = chiama_llm(storico_ordine, provider="openai", system_prompt=SYSTEM_PROMPT_ORDERBOT, temperature=0.0)

print("Riepilogo grezzo restituito dal modello:")
print(risultato_riepilogo["testo"])

---
## Cella 6 — TODO: valida il riepilogo JSON prima di usarlo

Come avvisato nella dispensa (sezione 4), un LLM può occasionalmente non
rispettare il formato richiesto. **Completa la funzione** `valida_riepilogo_ordine`
qui sotto: deve fare il parsing del testo con `json.loads`, gestire l'eventuale
errore, e verificare che tutte le chiavi richieste siano presenti.

In [ ]:
CHIAVI_RICHIESTE = {"pizze", "contorni", "bevande", "tipo_consegna", "indirizzo", "totale_stimato"}

def valida_riepilogo_ordine(testo_grezzo: str) -> dict | None:
    """Valida il riepilogo JSON restituito dal modello.

    Deve restituire:
    - il dizionario Python risultante dal parsing, se tutto è valido
    - None se il parsing fallisce o mancano chiavi richieste (stampando un messaggio d'errore)
    """
    # TODO 1: prova a fare il parsing di `testo_grezzo` con json.loads dentro un try/except.
    #         Se fallisce, stampa un messaggio d'errore chiaro e restituisci None.

    # TODO 2: verifica che tutte le chiavi in CHIAVI_RICHIESTE siano presenti nel
    #         dizionario ottenuto. Se manca qualcosa, stampa quali chiavi mancano
    #         e restituisci None.

    # TODO 3: verifica anche che, se tipo_consegna == "domicilio", il campo
    #         indirizzo non sia null/None. Se lo è, stampa un avviso e restituisci None.

    # TODO 4: se tutti i controlli passano, restituisci il dizionario validato.
    pass


ordine_validato = valida_riepilogo_ordine(risultato_riepilogo["testo"])
if ordine_validato:
    print("Ordine validato con successo:")
    print(json.dumps(ordine_validato, indent=2, ensure_ascii=False))
else:
    print("Validazione fallita: vedi i messaggi sopra.")

---
## Reto finale — Chatbot da riga di comando con storico e costo

Costruisci un OrderBot interattivo che:
- Mantenga lo storico della conversazione tra i turni
- Mostri il costo accumulato ad ogni turno
- Alla fine (`fine`/`salta`/`exit`) generi e validi il riepilogo JSON

> **Nota:** questa cella usa `input()`, che in alcuni ambienti Jupyter non
> funziona bene in modalità non interattiva. Se hai problemi, copia il codice
> in un file `orderbot.py` ed eseguilo da terminale con `python orderbot.py`.

In [ ]:
def orderbot_interattivo(provider: str = "openai") -> None:
    """Chatbot interattivo per prendere ordini in pizzeria."""
    storico: list[dict] = []
    costo_accumulato = 0.0

    print("OrderBot pronto. Scrivi 'fine' per chiudere l'ordine.\n")

    while True:
        messaggio = input("Tu: ").strip()
        if not messaggio:
            continue
        if messaggio.lower() in ("fine", "salta", "exit", "quit"):
            break

        storico.append({"role": "user", "content": messaggio})
        risultato = chiama_llm(storico, provider=provider, system_prompt=SYSTEM_PROMPT_ORDERBOT, temperature=0.3)
        storico.append({"role": "assistant", "content": risultato["testo"]})

        costo_turno = calcola_costo(risultato["tokens_input"], risultato["tokens_output"], risultato["modello"])
        costo_accumulato += costo_turno["costo_totale_usd"]

        print(f"\nOrderBot: {risultato['testo']}")
        print(f"[costo turno: ${costo_turno['costo_totale_usd']:.6f} | totale: ${costo_accumulato:.6f}]\n")

    # Genera e valida il riepilogo finale
    storico.append({"role": "user", "content": ISTRUZIONE_RIEPILOGO})
    risultato_finale = chiama_llm(storico, provider=provider, system_prompt=SYSTEM_PROMPT_ORDERBOT, temperature=0.0)

    ordine = valida_riepilogo_ordine(risultato_finale["testo"])
    if ordine:
        print("\nOrdine finale:")
        print(json.dumps(ordine, indent=2, ensure_ascii=False))
    else:
        print("\nNon è stato possibile validare il riepilogo. Testo grezzo:")
        print(risultato_finale["testo"])

    print(f"\nCosto totale della sessione: ${costo_accumulato:.6f}")


# Decommenta per eseguire in modo interattivo (o copia in orderbot.py e lancialo da terminale)
# orderbot_interattivo(provider="openai")

### Estensioni opzionali del reto (se hai tempo)

1. **Finestra scorrevole:** limita lo storico inviato al modello agli ultimi
   N turni, come discusso nella dispensa (sezione 5), e verifica che il bot
   continui a funzionare correttamente per un ordine tipico (che di solito
   dura pochi turni).
2. **Ricalcolo deterministico del totale:** invece di fidarti di
   `totale_stimato` generato dal modello, scrivi una funzione Python che
   ricalcola il prezzo sommando i valori di `MENU_PIZZERIA` in base agli
   articoli effettivamente ordinati, e confrontalo con la stima del modello.
3. **Cambio di provider a caldo:** aggiungi un comando speciale
   `/provider openai` o `/provider anthropic` che cambi provider senza
   perdere lo storico della conversazione.

In [ ]:
# Spazio per le estensioni opzionali


---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 10 — Modulo 10 del Corso Completo di IA Generativa*